# 개별종목 조합F — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합F 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합F의 피처 값만 지정합니다.
import json

COMBINATION = 'F'
FEATURE_COLUMNS = (
    'ret_5_rank',
    'sector_relative_rank',
    'turnover_rank',
    'hv_20_rank',
    'market_cap_percentile',
    'sector_market_cap_rank',
    'industry_stock_rank',
    'volume_z_20',
    'bb_position',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합F 피처: ('ret_5_rank', 'sector_relative_rank', 'turnover_rank', 'hv_20_rank', 'market_cap_percentile', 'sector_market_cap_rank', 'industry_stock_rank', 'volume_z_20', 'bb_position')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.3680,0.5012,-0.1332,0.3488,0.3540,0.0322,0.3612,0.3470,0.3543
1,2,balanced,980,20150123,20150421,0.3781,0.3978,-0.0197,0.3738,0.3745,0.0628,0.3680,0.3494,0.3666
2,3,balanced,1210,20151228,20160328,0.3484,0.3762,-0.0277,0.3449,0.3458,0.0217,0.3516,0.3720,0.3547
3,4,balanced,1439,20161202,20170228,0.3887,0.4617,-0.0730,0.3748,0.3775,0.0702,0.3743,0.3704,0.3778
4,5,balanced,1669,20171113,20180207,0.3456,0.3901,-0.0445,0.3383,0.3384,0.0093,0.3468,0.3118,0.3312
5,6,balanced,1899,20181024,20190118,0.3743,0.3725,0.0018,0.3684,0.3704,0.0574,0.3771,0.3556,0.3659
6,7,balanced,2129,20190930,20191224,0.3904,0.4781,-0.0878,0.3668,0.3702,0.0605,0.3775,0.3598,0.3719
7,8,balanced,2359,20200902,20201130,0.3602,0.3476,0.0126,0.3590,0.3646,0.0461,0.3600,0.3960,0.3710
8,9,balanced,2589,20210806,20211105,0.3815,0.3914,-0.0099,0.3716,0.3766,0.0644,0.3765,0.3386,0.3629
9,10,balanced,2818,20220714,20221012,0.3502,0.3454,0.0048,0.3441,0.3446,0.0198,0.3520,0.3575,0.3505


,OOS 폴드 평균
accuracy,0.3698
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0270
macro_f1,0.3612
balanced_accuracy,0.3638
mcc,0.0476
pr_auc_macro_ovr,0.3655
down_recall,0.3609
core_harmonic_mean,0.3636


재실행 명령: python scripts/run_stock_model_experiment.py
